# 24 - The drive-over decision from USS-prompted VLM grounding

Notebook 23: with the USS crop, Qwen + SAM3 find the relevant obstacle
significantly more often than from the full frame (+7 pt on unseen types).
This notebook makes the paper's actual call - Low (drive over) vs High
(stop) - from the grounded object, two ways:

- **direct**: Qwen answers `drive_over` true/false in the same call
- **lexicon**: Qwen's noun mapped to a height bin (`name_to_bin`)

each with the USS crop (`uss_*`) and from the full frame (`nouss_*`),
voted per sequence over its frames. Baselines on the paper's test split: the
**production USS height classifier** and the **trained detector** (direction-
correct camera, majority vote). Adds low types (woodenboard, speedbump) and
more high types (dummychild, bicyclestand) so Low/High is measurable.

Also writes `frames.csv` for the gold review (`scripts/build_plat_gold_review.py`).
**GPU: L4 or A100.** Qwen only (no SAM3), resumable.

In [ ]:
# --- pipeline bootstrap (identical first cell in every pipeline notebook) ---
import subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/vision-uss-research")
    assert (REPO_DIR / ".git").exists(), \
        "repo not found - run scripts/bootstrap_colab.sh first (set REPO_URL)"
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(REPO_DIR / "requirements" / "colab.txt")], check=True)
    # canmatrix 1.3.0 (2026-10-01) breaks `import asammdf` (TypeError in
    # asammdf/blocks/types.py) - pin until asammdf supports it
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "asammdf==8.8.27",
                    "canmatrix==1.2"], check=True)
else:
    REPO_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / "pyproject.toml").exists())

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

from vision_uss_research.runs import git_sha
from vision_uss_research.settings import load_paths

PROFILE = "colab_drive" if IN_COLAB else "local"
ARTIFACTS_OVERRIDE = None
RAW_ROOT_OVERRIDE = None

paths = load_paths(profile=PROFILE)
ARTIFACTS = Path(ARTIFACTS_OVERRIDE or paths.outputs_dir)
RAW_ROOT = Path(RAW_ROOT_OVERRIDE or paths.raw_data_root)
print(f"repo      : {REPO_DIR}")
print(f"git sha   : {git_sha(REPO_DIR)}")
print(f"artifacts : {ARTIFACTS}")
print(f"raw root  : {RAW_ROOT}")

In [ ]:
import json as _json
import sys

import cv2
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

sys.path.insert(0, str(REPO_DIR / "scripts"))
import uss_prompt_analysis as upa
from vision_uss_research.labeling.corridor import pick_target_instance
from vision_uss_research.method.uss_prompt import (fit_ground_camera, region_box,
                                                   track_target_slots)
from vision_uss_research.method.vlm_prompt import (decision_prompt, name_to_bin,
                                                   padded_crop, parse_drive_over,
                                                   parse_name_box)
from vision_uss_research.qa.flags import box_iou
from vision_uss_research.runs import finish_run, new_run_id, start_run
from vision_uss_research.sequences import frames_window_id

DATASET_RUN = "20260718_dataset_v2"
TYPES_RUN = "dataset_v2daysplit"          # manifest carrying target_object per sequence
PROJ_RUN = "prompt_uss_proj_v1"
TYPES = ["cone", "bush", "bollard", "car", "pole", "dummychild", "bicyclestand",
         "curbstone", "woodenboard", "speedbump"]
LOW = {"curbstone", "woodenboard", "speedbump"}          # truth bin: < ~25 cm
UNSEEN = {"cone", "bush", "bollard", "car"}
SEQS_PER_TYPE = 30
FRAMES_PER_SEQ = 5
REGION = dict(half_w=160, below=100, above=250)
QWEN_MODEL = "Qwen/Qwen2.5-VL-7B-Instruct"
SCORE_THRESHOLD = 0.2
FOUND_IOU = 0.3
MIN_GT_H, MIN_GT_W = 15, 8
SEED = 0

MAN_DIR = ARTIFACTS / "datasets" / DATASET_RUN
PROJ_DIR = ARTIFACTS / "method" / PROJ_RUN
WINDOW_ROOT = ARTIFACTS / "frames" / frames_window_id(20, (0.30, 0.85))
RUN_DIR = ARTIFACTS / "method" / new_run_id("prompt", "decision_v1")
QWEN_CSV = RUN_DIR / "qwen.csv"
RUN_DIR.mkdir(parents=True, exist_ok=True)
KEYS = ["sequence_id", "camera", "frame_idx"]

slots, boxes = upa.load(PROJ_DIR, MAN_DIR)
types = (pd.read_csv(ARTIFACTS / "datasets" / TYPES_RUN / "dataset_manifest.csv",
                     usecols=["sequence_id", "target_object"]).drop_duplicates("sequence_id"))
pairs = (slots.merge(track_target_slots(slots), on=["sequence_id", "camera", "slot"])
         .merge(boxes, on=KEYS).merge(types, on="sequence_id"))
pairs["day"] = pairs.sequence_id.str.split("_").str[1]
pairs = pairs[((pairs.y1 - pairs.y0) >= MIN_GT_H) & ((pairs.x1 - pairs.x0) >= MIN_GT_W)]

# cross-fit calibration: two folds per (camera, day), global per fold as fallback
seq_fold = {}
for (_, _), d in pairs.groupby(["camera", "day"]):
    for i, s in enumerate(sorted(d.sequence_id.unique())):
        seq_fold[s] = i % 2
pairs["fold"] = pairs.sequence_id.map(seq_fold)
def fit(d, cam_name):
    return fit_ground_camera(d.mid_x, d.mid_y, (d.x0 + d.x1) / 2, d.y1, cam_name)[0]
glob = {(c, f): fit(pairs[(pairs.camera == c) & (pairs.fold == f)], c)
        for c in ("front", "rear") for f in (0, 1)}
cams = {}
for (c, day, f), d in pairs.groupby(["camera", "day", "fold"]):
    if len(d) >= 25:
        cams[(c, day, f)] = fit(d, c)
print(f"per-day-fold calibrations: {len(cams)}")

rng = np.random.default_rng(SEED)
chosen = []
for t in TYPES:
    seqs = pairs[pairs.target_object == t].sequence_id.drop_duplicates()
    seqs = seqs.sample(min(60, len(seqs)), random_state=SEED).head(SEQS_PER_TYPE)
    sub = pairs[pairs.sequence_id.isin(seqs)].sample(frac=1, random_state=SEED)
    chosen.append(sub.groupby("sequence_id").head(FRAMES_PER_SEQ))
todo = pd.concat(chosen, ignore_index=True)
proj = []
for r in todo.itertuples(index=False):
    other = 1 - r.fold
    m = cams.get((r.camera, r.day, other))
    src = "day"
    if m is None:
        m, src = glob[(r.camera, other)], "global"
    u, v, ok = m.project(r.mid_x, r.mid_y)
    proj.append((float(u), float(v), bool(ok), src))
todo["pu"], todo["pv"], todo["proj_ok"], todo["calib"] = zip(*proj)
todo = todo[todo.proj_ok]

todo["item"] = todo.sequence_id + "|" + todo.camera + "|" + todo.frame_idx.astype(str)
start_run(RUN_DIR, config={"types": TYPES, "seqs_per_type": SEQS_PER_TYPE, "qwen": QWEN_MODEL,
                           "region": REGION, "found_iou": FOUND_IOU},
          inputs=[f"method/{PROJ_RUN}", f"datasets/{DATASET_RUN}", f"datasets/{TYPES_RUN}"],
          repo_root=REPO_DIR)
print(todo.groupby("target_object").agg(seqs=("sequence_id", "nunique"),
                                         frames=("frame_idx", "size")).to_string())
print(f"calibration: {todo.calib.value_counts().to_dict()} | todo frames {len(todo)}")
todo.to_csv(RUN_DIR / "frames.csv", index=False)   # for scripts/build_plat_gold_review.py

## Qwen: name, box, drive-over

In [ ]:
# ---- Qwen: name + box + drive-over judgment, on the USS crop and on the full frame
import subprocess
import torch

def run_stage(path, fn, every=25):
    done = pd.read_csv(path) if path.exists() else pd.DataFrame(columns=["item"])
    rest = todo[~todo.item.isin(set(done["item"]))]
    buf = []
    for k, r in enumerate(tqdm(rest.itertuples(index=False), total=len(rest), unit="frame"), 1):
        buf.append({"item": r.item, **fn(r)})
        if len(buf) >= every or k == len(rest):
            done = pd.concat([done, pd.DataFrame(buf)], ignore_index=True)
            done.to_csv(path, index=False); buf = []
    return pd.read_csv(path)

if not (QWEN_CSV.exists() and len(pd.read_csv(QWEN_CSV)) >= len(todo)):
    if IN_COLAB:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "qwen-vl-utils"], check=True)
    from qwen_vl_utils import process_vision_info
    from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    qmodel = Qwen2_5_VLForConditionalGeneration.from_pretrained(QWEN_MODEL, torch_dtype=dtype,
                                                                device_map="auto")
    qproc = AutoProcessor.from_pretrained(QWEN_MODEL)

    def ask(pil, view):
        msgs = [{"role": "user", "content": [{"type": "image", "image": pil},
                                             {"type": "text", "text": decision_prompt(view)}]}]
        text = qproc.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        image_inputs, video_inputs = process_vision_info(msgs)
        inputs = qproc(text=[text], images=image_inputs, videos=video_inputs,
                       padding=True, return_tensors="pt").to(qmodel.device)
        _, gh, gw = inputs["image_grid_thw"][0].tolist()
        with torch.inference_mode():
            out = qmodel.generate(**inputs, max_new_tokens=80, do_sample=False)
        reply = qproc.batch_decode(out[:, inputs["input_ids"].shape[1]:],
                                   skip_special_tokens=True)[0]
        name, box = parse_name_box(reply, (gw * 14, gh * 14), pil.size)
        return name, box, reply.strip()[:200], parse_drive_over(reply)

    def qwen_one(r):
        p = WINDOW_ROOT / r.sequence_id / r.camera / f"{int(r.frame_idx):05d}.jpg"
        if not p.exists():
            return {"error": "no_frame"}
        img = Image.open(p).convert("RGB")
        W, H = img.size
        reg = region_box(r.pu, r.pv, W, H, **REGION)
        crop = padded_crop(reg, W, H)
        row = {"error": "", "crop": "" if crop is None else ",".join(map(str, crop))}
        views = [("full", img, (0, 0))]
        if crop is None:                      # USS object projects off-frame: USS arm misses
            row.update(crop_name=None, crop_reply="region_off_image", crop_drive_over=None,
                       crop_qx0=np.nan, crop_qy0=np.nan, crop_qx1=np.nan, crop_qy1=np.nan)
        else:
            views.insert(0, ("crop", img.crop(crop), crop[:2]))
        for view, pil, off in views:
            name, box, reply, drive = ask(pil, view)
            row[f"{view}_name"] = name
            row[f"{view}_drive_over"] = drive
            row[f"{view}_reply"] = reply
            if box is not None:
                box = box + np.array([off[0], off[1], off[0], off[1]])
            for c, v in zip(("x0", "y0", "x1", "y1"),
                            box if box is not None else [np.nan] * 4):
                row[f"{view}_q{c}"] = float(v)
        return row

    qwen = run_stage(QWEN_CSV, qwen_one)
    del qmodel
    torch.cuda.empty_cache()
qwen = pd.read_csv(QWEN_CSV)
print(f"Qwen rows {len(qwen)} | crop named {qwen.crop_name.notna().mean():.0%} | "
      f"full named {qwen.full_name.notna().mean():.0%}")

## Decisions vs production USS and the trained detector

In [ ]:
res = todo.merge(qwen, on="item", suffixes=("", "_q"))
res = res[res.error.fillna("") == ""].copy()
res["truth"] = np.where(res.target_object.isin(LOW), "low", "high")
for v in ("crop", "full"):
    d = res[f"{v}_drive_over"]
    res[f"{v}_direct"] = np.where(d.astype(str) == "True", "low",
                          np.where(d.astype(str) == "False", "high", None))
    res[f"{v}_lexicon"] = [name_to_bin(n) for n in res[f"{v}_name"]]

def vote(s):
    s = s.dropna()
    return s.mode().iat[0] if len(s) else None
ARMS = {"uss_direct": "crop_direct", "nouss_direct": "full_direct",
        "uss_lexicon": "crop_lexicon", "nouss_lexicon": "full_lexicon"}
seq = res.groupby("sequence_id").agg(target_object=("target_object", "first"),
                                     truth=("truth", "first"),
                                     **{a: (c, vote) for a, c in ARMS.items()}).reset_index()

# baselines on the paper's test split: production USS classifier + trained
# detector (direction-correct camera, majority vote - the fixed protocol)
fs = pd.read_csv(ARTIFACTS / "reports" / "fusion_camera_uss" / "fusion_signals.csv",
                 usecols=["sequence_id", "uss_bin"])
fp = pd.read_csv(ARTIFACTS / "reports" / "eval_causal" / "frame_predictions.csv",
                 usecols=["sequence_id", "pred_bin"])
det = fp.groupby("sequence_id").pred_bin.agg(lambda s: s.mode().iat[0]).rename("detector")
test_ids = set(boxes.loc[boxes.split == "test", "sequence_id"])
seq = seq.merge(fs, on="sequence_id", how="left").join(det, on="sequence_id")
seq["in_test"] = seq.sequence_id.isin(test_ids)
seq.loc[~seq.in_test, ["uss_bin", "detector"]] = None   # trained detector saw the train split
seq = seq.rename(columns={"uss_bin": "uss_classifier"})

METHODS = list(ARMS) + ["uss_classifier", "detector"]
def score(d):
    out = {"seqs": len(d)}
    for m in METHODS:
        dec = d[m]
        has = dec.notna()
        ok = (dec == d.truth) & has
        out[f"{m}_acc_decided"] = round(ok[has].mean(), 3) if has.any() else None
        out[f"{m}_acc_all"] = round(ok.mean(), 3)
        hi = d.truth == "high"
        out[f"{m}_false_low"] = round(((dec == "low") & hi).sum() / max(hi.sum(), 1), 3)
    return pd.Series(out)
pd.set_option("display.width", 300); pd.set_option("display.max_columns", 40)
by_type = seq.groupby("target_object").apply(score, include_groups=False)
print("per type, all sequences (accuracy counting abstentions as wrong):")
display(by_type[["seqs"] + [f"{m}_acc_all" for m in METHODS]])
overall = pd.DataFrame({"all": score(seq), "test split": score(seq[seq.in_test])}).T
display(overall)
lowhi = seq.assign(low=seq.truth == "low")
bal = {m: round(np.mean([((g[m] == g.truth)).mean() for _, g in lowhi.groupby("low")]), 3)
       for m in METHODS}
print("balanced accuracy over all sequences (abstain = wrong):", bal)
print("\nQwen names for LOW types (crop view):")
for t in sorted(LOW):
    print(f"  {t:12s} {res[res.target_object == t].crop_name.value_counts().head(5).to_dict()}")
res.to_csv(RUN_DIR / "scored.csv", index=False)
seq.to_csv(RUN_DIR / "sequences.csv", index=False)
overall.to_csv(RUN_DIR / "summary.csv")
by_type.to_csv(RUN_DIR / "summary_by_type.csv")
finish_run(RUN_DIR, summary={"balanced_accuracy": bal,
                             "overall": overall.reset_index().to_dict(orient="records")})
print(f"download {RUN_DIR.name}/ (summary*.csv, sequences.csv, frames.csv)")